In [3]:
# ==============================
# STEP 1: IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import re
import torch

from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

# ==============================
# STEP 2: LOAD DATASET
# ==============================
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

# ==============================
# STEP 3: ADD LABELS
# ==============================
fake["label"] = 0
true["label"] = 1

# ==============================
# STEP 4: COMBINE + REDUCE DATA (FAST)
# ==============================
df = pd.concat([fake, true])
df = df.sample(frac=1).reset_index(drop=True)

# 🔥 Reduce dataset size (IMPORTANT)
df = df.sample(2000)

# ==============================
# STEP 5: SELECT COLUMNS
# ==============================
df = df[["text", "label"]]

# ==============================
# STEP 6: CLEAN TEXT
# ==============================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df["text"] = df["text"].apply(clean_text)

# ==============================
# STEP 7: TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

# ==============================
# STEP 8: LOAD FAST MODEL
# ==============================
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# ==============================
# STEP 9: TOKENIZATION (SHORTER LENGTH)
# ==============================
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)

# ==============================
# STEP 10: DATASET CLASS
# ==============================
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, y_train)
test_dataset = NewsDataset(test_encodings, y_test)

# ==============================
# STEP 11: TRAIN MODEL (FAST SETTINGS)
# ==============================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
    save_steps=500,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

# ==============================
# STEP 12: EVALUATE
# ==============================
trainer.evaluate()

# ==============================
# STEP 13: PREDICTION FUNCTION
# ==============================
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    label = "Real" if torch.argmax(probs)==1 else "Fake"
    confidence = float(torch.max(probs))

    return label, confidence

# ==============================
# STEP 14: TEST
# ==============================
print(predict("Breaking news: Government announces new policy"))

# ==============================
# STEP 15: SAVE MODEL
# ==============================
model.save_pretrained("fast_fake_news_model")
tokenizer.save_pretrained("fast_fake_news_model")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1779.34it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\ADMIN\AppData\Local\Pro

Step,Training Loss
50,0.197136
100,0.029622


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
0.029622,0.036710,100


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17952\1423986647.py:124: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  confidence = float(torch.max(probs))


('Fake', 0.992144763469696)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


('fast_fake_news_model\\tokenizer_config.json',
 'fast_fake_news_model\\tokenizer.json')